# Stage 7 — Model-free process indicators vs AI use (Fay-BRR, 80 replicates). CPU only, ~5–10 min.
Run in the FIRST Google account (it holds PISA2025_stage1 and PISA2025_extract). Runtime type: CPU.

In [ ]:
import os, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
S1='/content/drive/MyDrive/PISA2025_stage1'; SX='/content/drive/MyDrive/PISA2025_extract'; OUT='/content/drive/MyDrive/PISA2025_stage7'; os.makedirs(OUT,exist_ok=True)
CNTS=['ARM','KAZ','MDA','LVA','LTU','EST','JPN','ARE','SGP']
PRED=['AIUSESCH','ST438Q01DA','ST438Q02DA','ST438Q03DA','ST438Q04DA']
IND=['acc','rapid_rate','noaction_rate','notreached_rate','log_F','log_T']
def student_table(dom):
    L=[]
    for c in CNTS:
        d=pd.read_parquet(f'{S1}/long_{dom}_{c}.parquet'); d=d[d.dich]
        e=d[~d.not_reached & d.valid_t & ~d.trim]
        fitem=e.item.map(e.groupby('item').f.apply(lambda s: s.notna().mean()>0.5))   # items that record time-to-first-action
        g=e.assign(rapid=e.rapid.astype(float),noact=np.where(fitem,e.f.isna(),np.nan).astype(float),lf=np.log(e.f.clip(lower=0.1)),lt=np.log(e.t)).groupby('CNTSTUID').agg(
            acc=('x','mean'),rapid_rate=('rapid','mean'),noaction_rate=('noact','mean'),log_F=('lf','mean'),log_T=('lt','mean'),n_items=('x','size'))
        g['notreached_rate']=d.groupby('CNTSTUID').not_reached.mean()
        cov=d.drop_duplicates('CNTSTUID').set_index('CNTSTUID')[['CNT','ESCS','ST004D01T','W_FSTUWT']+PRED]
        rw=pd.read_parquet(f'{SX}/STU_{c}.parquet',columns=['CNTSTUID']+[f'W_FSTURWT{j}' for j in range(1,81)]).set_index('CNTSTUID')
        L.append(g.join(cov).join(rw))
    return pd.concat(L).reset_index()
def brr(P,pred,y,armonly=False):
    Q=P.dropna(subset=[pred,'ESCS','ST004D01T',y,'W_FSTURWT1'])
    if armonly: Q=Q[Q.CNT=='ARM']
    if len(Q)<100: return {'b':np.nan,'se_BRR':np.nan,'z':np.nan,'std_beta':np.nan,'n':len(Q)}
    X=pd.get_dummies(Q[[pred,'ESCS','ST004D01T']+([] if armonly else ['CNT'])],columns=['ST004D01T']+([] if armonly else ['CNT']),drop_first=True,dtype=float)
    X.insert(0,'const',1.); X=X.values; yy=Q[y].values; cnt=Q.CNT.values
    def est(w): W=w/pd.Series(w).groupby(cnt).transform('mean').values; XtW=X.T*W; return np.linalg.lstsq(XtW@X,XtW@yy,rcond=None)[0][1]
    b=est(Q.W_FSTUWT.values); br=np.array([est(Q[f'W_FSTURWT{j}'].values) for j in range(1,81)]); se=np.sqrt(((br-b)**2).sum()/20.)   # Fay k=0.5, 80 replicates
    sx=Q[pred].std(); sy=Q[y].std()
    return {'b':b,'se_BRR':se,'z':b/se,'std_beta':b*sx/sy,'n':len(Q)}
rows=[]; desc=[]
for dom in ['math','read']:
    P=student_table(dom); P.to_csv(f'{OUT}/students_{dom}.csv',index=False)
    for c,g in P.groupby('CNT'):
        w=g.W_FSTUWT; desc.append({'domain':dom,'CNT':c,'n':len(g),'AI_cov':round(g.AIUSESCH.notna().mean(),3),**{k:round(float(np.average(g[k].fillna(g[k].mean()),weights=w)),4) for k in IND}})
    for pred in PRED:
        for y in IND:
            for scope in ['pooled_9','ARM_only']:
                r=brr(P,pred,y,armonly=(scope=='ARM_only')); rows.append({'domain':dom,'predictor':pred,'outcome':y,'scope':scope,**r})
R=pd.DataFrame(rows).round(4); R.to_csv(f'{OUT}/stage7_brr_indicators.csv',index=False)
Dd=pd.DataFrame(desc); Dd.to_csv(f'{OUT}/stage7_descriptives.csv',index=False)
pd.set_option('display.width',200); print(Dd.to_string())
print(R[R.predictor=='AIUSESCH'].to_string()); print('STAGE7 DONE')
